# THEIA · Pipeline fuera de la app

Notebook para probar el pipeline completo (detección → pose → filtro estadístico → GPA/PCA) sin abrir la app móvil. Está pensado como banco de pruebas para modelos TFLite y para comparar resultados (LM, GPA, PCA) con tus referencias en R.


## Contenido
- Configuración y utilidades compartidas.
- Flujo de inferencia (detector nano + pose medium) con el mismo letterboxing/cropping que en la app.
- Filtro estadístico de landmarks usando `assets/keypoint_stats_model.json` y exportación CSV.
- GPA + PCA reproducidos en Python (idénticos a `MorphometricAnalysis` en Dart).
- Gráficos: wireframes ±2DE, tabla de scores y morfoespacio PC1–PC2.


## Requisitos previos
1. Python ≥ 3.10.
2. Dependencias recomendadas: `tflite-runtime` (o `tensorflow>=2.12`), `numpy`, `pandas`, `matplotlib`, `Pillow`.
3. Ubica las imágenes a evaluar en una carpeta y actualiza la variable `IMAGES_DIR` más abajo.
4. Los modelos `.tflite` y el archivo de estadísticas se toman directamente de `assets/`.


In [ ]:
%matplotlib inline
import json
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

try:
    from tflite_runtime.interpreter import Interpreter  # type: ignore
    TFLITE_SOURCE = "tflite-runtime"
except ImportError:
    try:
        import tensorflow as tf
    except ImportError as exc:
        raise ImportError(
            "Instala tensorflow>=2.12 o tflite-runtime para ejecutar las inferencias."
        ) from exc
    else:
        Interpreter = tf.lite.Interpreter  # type: ignore
        TFLITE_SOURCE = "tensorflow"

plt.style.use("seaborn-v0_8")
print(f"Backend TFLite: {TFLITE_SOURCE}")


## Paths y parámetros globales
Ajusta `IMAGES_DIR` para apuntar a tus especímenes. Si ejecutas el notebook desde otra carpeta, el helper `find_repo_root` localizará automáticamente la raíz del proyecto (busca `pubspec.yaml`).


In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pubspec.yaml").exists():
            return candidate
    raise RuntimeError("No se encontró la raíz del repo (pubspec.yaml)")

PROJECT_ROOT = find_repo_root(Path.cwd())
ASSETS_DIR = PROJECT_ROOT / "assets"
IMAGES_DIR = PROJECT_ROOT / "data" / "specimens_demo"  # <-- Actualiza aquí
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "offline_pipeline"

MODELS = {
    "detector": ASSETS_DIR / "detector_nano_fp32.tflite",
    "pose": ASSETS_DIR / "pose_medium_fp32.tflite",
    "stats": ASSETS_DIR / "keypoint_stats_model.json",
}

MODEL_INPUT_PIXELS = 640
DETECTIONS = 8400
CONFIDENCE_INDEX = 4
KEYPOINTS_START_INDEX = 5
KEYPOINTS_TOTAL = 32
CROP_PADDING_RATIO = 0.15
MIN_DETECT_CONFIDENCE = 0.40
MIN_POSE_CONFIDENCE = 0.25
DEFAULT_STD_TOLERANCE = 3.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Root: {PROJECT_ROOT}")
print(f"Imágenes: {IMAGES_DIR}")
print(f"Exportaciones: {OUTPUT_DIR}")


## Modelo estadístico para el filtro estructural
La lógica replica `_validateAndParseResult` de `BatchModeScreen`: para cada punto se evalúa la distancia (en coordenadas relativas al centro de la caja) respecto a la media ± N desviaciones estándar. Puedes personalizar tolerancias por punto editando `CUSTOM_STD_TOLERANCES`.


In [ ]:
with open(MODELS["stats"], "r", encoding="utf-8") as fh:
    stats_payload = json.load(fh)

STATS_MODEL: Dict[int, Dict[str, float]] = {}
for row in stats_payload:
    idx = int(row["keypoint_index"])
    STATS_MODEL[idx] = {
        "mean_x": float(row["mean_x"]),
        "mean_y": float(row["mean_y"]),
        "std_dev_x": float(row["std_dev_x"]),
        "std_dev_y": float(row["std_dev_y"]),
    }

CUSTOM_STD_TOLERANCES: Dict[int, float] = {}
print(f"Entradas en modelo estadístico: {len(STATS_MODEL)} puntos")


## Utilidades de inferencia y geometría
Incluye letterboxing idéntico a Android, conversión de cajas, expansión del recorte y reconstrucción de keypoints normalizados.


In [ ]:
@dataclass
class LetterboxParams:
    scale: float
    pad_x: float
    pad_y: float

@dataclass
class DetectionBox:
    left: float
    top: float
    right: float
    bottom: float
    confidence: float

@dataclass
class CropBounds:
    left: int
    top: int
    width: int
    height: int

@dataclass
class CropInfo:
    left: float
    top: float
    width: float
    height: float


def load_interpreter(model_path: Path) -> Interpreter:
    if not model_path.exists():
        raise FileNotFoundError(model_path)
    interpreter = Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    return interpreter


def letterbox_image(image: Image.Image, target_size: int = MODEL_INPUT_PIXELS) -> Tuple[np.ndarray, LetterboxParams]:
    image = image.convert("RGB")
    width, height = image.size
    scale = min(target_size / width, target_size / height)
    scaled_w = max(1, int(round(width * scale)))
    scaled_h = max(1, int(round(height * scale)))
    resized = image.resize((scaled_w, scaled_h), Image.BILINEAR)
    pad_x = (target_size - scaled_w) / 2.0
    pad_y = (target_size - scaled_h) / 2.0
    canvas = Image.new("RGB", (target_size, target_size), (0, 0, 0))
    canvas.paste(resized, (int(round(pad_x)), int(round(pad_y))))
    arr = np.asarray(canvas, dtype=np.float32) / 255.0
    arr = np.expand_dims(arr, axis=0)
    params = LetterboxParams(scale=float(scale), pad_x=float(pad_x), pad_y=float(pad_y))
    return arr, params


def letterbox_to_original(value: float, pad: float, scale: float, limit: float) -> float:
    if scale == 0:
        return 0.0
    out = (value - pad) / scale
    return float(np.clip(out, 0.0, limit))


def _transpose_output(raw: np.ndarray) -> np.ndarray:
    squeezed = np.squeeze(raw, axis=0)
    if squeezed.ndim != 2:
        squeezed = squeezed.reshape(squeezed.shape[0], -1)
    rows, cols = squeezed.shape
    if rows == DETECTIONS:
        return squeezed
    if cols == DETECTIONS:
        return squeezed.T
    return squeezed


def run_model(interpreter: Interpreter, input_tensor: np.ndarray, min_conf: float) -> Optional[np.ndarray]:
    input_details = interpreter.get_input_details()[0]
    interpreter.set_tensor(input_details["index"], input_tensor.astype(np.float32))
    interpreter.invoke()
    output_details = interpreter.get_output_details()[0]
    raw = interpreter.get_tensor(output_details["index"])
    detections = _transpose_output(raw)
    if detections.shape[1] <= CONFIDENCE_INDEX:
        return None
    confidences = detections[:, CONFIDENCE_INDEX]
    best_idx = int(np.argmax(confidences))
    best_conf = float(confidences[best_idx])
    if best_conf < min_conf:
        return None
    return detections[best_idx]


def convert_to_detection_box(vec: np.ndarray, letterbox: LetterboxParams, width: int, height: int) -> Optional[DetectionBox]:
    cx, cy, w_box, h_box = map(float, vec[:4])
    if w_box <= 0 or h_box <= 0:
        return None
    x1_letter = cx - w_box / 2.0
    y1_letter = cy - h_box / 2.0
    x2_letter = cx + w_box / 2.0
    y2_letter = cy + h_box / 2.0
    x1 = letterbox_to_original(x1_letter, letterbox.pad_x, letterbox.scale, width)
    y1 = letterbox_to_original(y1_letter, letterbox.pad_y, letterbox.scale, height)
    x2 = letterbox_to_original(x2_letter, letterbox.pad_x, letterbox.scale, width)
    y2 = letterbox_to_original(y2_letter, letterbox.pad_y, letterbox.scale, height)
    left = max(0.0, min(x1, x2))
    right = min(float(width), max(x1, x2))
    top = max(0.0, min(y1, y2))
    bottom = min(float(height), max(y1, y2))
    if right - left <= 1.0 or bottom - top <= 1.0:
        return None
    return DetectionBox(left=left, top=top, right=right, bottom=bottom, confidence=float(vec[CONFIDENCE_INDEX]))


def expand_crop_bounds(box: DetectionBox, width: int, height: int, padding_ratio: float = CROP_PADDING_RATIO) -> Optional[CropBounds]:
    bw = box.right - box.left
    bh = box.bottom - box.top
    if bw <= 0 or bh <= 0:
        return None
    pad_x = bw * padding_ratio
    pad_y = bh * padding_ratio
    left = max(0.0, box.left - pad_x)
    top = max(0.0, box.top - pad_y)
    right = min(float(width), box.right + pad_x)
    bottom = min(float(height), box.bottom + pad_y)
    left_i = int(math.floor(left))
    top_i = int(math.floor(top))
    right_i = int(math.ceil(right))
    bottom_i = int(math.ceil(bottom))
    right_i = min(right_i, width)
    bottom_i = min(bottom_i, height)
    if right_i - left_i < 2 or bottom_i - top_i < 2:
        return None
    return CropBounds(left=left_i, top=top_i, width=right_i - left_i, height=bottom_i - top_i)


def build_pose_result(vec: np.ndarray, letterbox: LetterboxParams, crop: CropInfo, width: int, height: int) -> Optional[Dict[str, object]]:
    if len(vec) < KEYPOINTS_START_INDEX + KEYPOINTS_TOTAL * 3:
        return None
    x1_letter = vec[0] - vec[2] / 2.0
    y1_letter = vec[1] - vec[3] / 2.0
    x2_letter = vec[0] + vec[2] / 2.0
    y2_letter = vec[1] + vec[3] / 2.0
    x1_crop = letterbox_to_original(x1_letter, letterbox.pad_x, letterbox.scale, crop.width)
    y1_crop = letterbox_to_original(y1_letter, letterbox.pad_y, letterbox.scale, crop.height)
    x2_crop = letterbox_to_original(x2_letter, letterbox.pad_x, letterbox.scale, crop.width)
    y2_crop = letterbox_to_original(y2_letter, letterbox.pad_y, letterbox.scale, crop.height)
    x1 = np.clip(x1_crop + crop.left, 0.0, width)
    y1 = np.clip(y1_crop + crop.top, 0.0, height)
    x2 = np.clip(x2_crop + crop.left, 0.0, width)
    y2 = np.clip(y2_crop + crop.top, 0.0, height)
    if x2 - x1 <= 1.0 or y2 - y1 <= 1.0:
        return None
    box = [
        float(((x1 + x2) / 2.0) / width),
        float(((y1 + y2) / 2.0) / height),
        float((x2 - x1) / width),
        float((y2 - y1) / height),
    ]
    keypoints: List[Tuple[float, float, float]] = []
    for i in range(KEYPOINTS_TOTAL):
        idx = KEYPOINTS_START_INDEX + i * 3
        raw_x, raw_y, conf = float(vec[idx]), float(vec[idx + 1]), float(vec[idx + 2])
        crop_x = letterbox_to_original(raw_x, letterbox.pad_x, letterbox.scale, crop.width)
        crop_y = letterbox_to_original(raw_y, letterbox.pad_y, letterbox.scale, crop.height)
        abs_x = np.clip(crop_x + crop.left, 0.0, width)
        abs_y = np.clip(crop_y + crop.top, 0.0, height)
        keypoints.append((float(abs_x / width), float(abs_y / height), conf))
    return {
        "box": box,
        "keypoints": keypoints,
        "pose_confidence": float(vec[CONFIDENCE_INDEX]),
    }


## Carga de intérpretes y helpers del pipeline


In [ ]:
detector_interpreter = load_interpreter(MODELS["detector"])
pose_interpreter = load_interpreter(MODELS["pose"])
print("Modelos listos ✅")


In [ ]:
def run_prediction_pipeline(image_path: Path) -> Tuple[Optional[Dict[str, object]], Optional[str]]:
    image = Image.open(image_path)
    original_w, original_h = image.size

    detector_input, detector_letterbox = letterbox_image(image)
    detection_vec = run_model(detector_interpreter, detector_input, MIN_DETECT_CONFIDENCE)
    if detection_vec is None:
        return None, "Sin detección con confianza suficiente"

    detection_box = convert_to_detection_box(detection_vec, detector_letterbox, original_w, original_h)
    if detection_box is None:
        return None, "Caja inválida"

    crop_bounds = expand_crop_bounds(detection_box, original_w, original_h)
    if crop_bounds is None:
        return None, "Recorte inválido"

    crop = image.crop((
        crop_bounds.left,
        crop_bounds.top,
        crop_bounds.left + crop_bounds.width,
        crop_bounds.top + crop_bounds.height,
    ))

    crop_info = CropInfo(
        left=float(crop_bounds.left),
        top=float(crop_bounds.top),
        width=float(crop_bounds.width),
        height=float(crop_bounds.height),
    )

    pose_input, pose_letterbox = letterbox_image(crop)
    pose_vec = run_model(pose_interpreter, pose_input, MIN_POSE_CONFIDENCE)
    if pose_vec is None:
        return None, "Pose sin confianza suficiente"

    pose_result = build_pose_result(pose_vec, pose_letterbox, crop_info, original_w, original_h)
    if pose_result is None:
        return None, "Reconstrucción de keypoints fallida"

    pose_result["image_path"] = str(image_path)
    pose_result["image_name"] = image_path.name
    return pose_result, None


def batch_predict(image_paths: List[Path]) -> Tuple[List[Dict[str, object]], List[Dict[str, str]]]:
    predictions: List[Dict[str, object]] = []
    failures: List[Dict[str, str]] = []
    for path in image_paths:
        try:
            result, error = run_prediction_pipeline(path)
        except Exception as exc:  # pylint: disable=broad-except
            failures.append({"image_name": path.name, "reason": str(exc)})
            continue
        if error is not None or result is None:
            failures.append({"image_name": path.name, "reason": error or "Desconocido"})
        else:
            predictions.append(result)
    return predictions, failures


## Ejecutar inferencias sobre la carpeta seleccionada


In [ ]:
supported_ext = {".jpg", ".jpeg", ".png", ".bmp"}
if not IMAGES_DIR.exists():
    raise FileNotFoundError(f"Carpeta no encontrada: {IMAGES_DIR}")
image_paths = [p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in supported_ext]
image_paths.sort()
print(f"Encontradas {len(image_paths)} imágenes")

predictions, failures = batch_predict(image_paths)
print(f"Predicciones válidas: {len(predictions)}")
if failures:
    print(f"Fallos: {len(failures)} (ver tabla)")
    display(pd.DataFrame(failures))


## Filtro estadístico y exportación de landmarks aprobados


In [ ]:
def _tolerance_for(idx: int) -> float:
    return CUSTOM_STD_TOLERANCES.get(idx, DEFAULT_STD_TOLERANCE)


def validate_prediction(pred: Dict[str, object]) -> Tuple[bool, List[int]]:
    box_cx = pred["box"][0]
    box_cy = pred["box"][1]
    failed: List[int] = []
    for idx, (x_norm, y_norm, _conf) in enumerate(pred["keypoints"]):
        stats = STATS_MODEL.get(idx)
        if not stats:
            continue
        rel_x = x_norm - box_cx
        rel_y = y_norm - box_cy
        tol = _tolerance_for(idx)
        sx = stats["std_dev_x"]
        sy = stats["std_dev_y"]
        cond_x = sx > 0 and abs(rel_x - stats["mean_x"]) > tol * sx
        cond_y = sy > 0 and abs(rel_y - stats["mean_y"]) > tol * sy
        if cond_x or cond_y:
            failed.append(idx + 1)
    return len(failed) == 0, failed


def prediction_to_row(pred: Dict[str, object]) -> Dict[str, float]:
    row = {"image_name": pred["image_name"]}
    for i, (x_norm, y_norm, _conf) in enumerate(pred["keypoints"], start=1):
        row[f"kpt{i}_x"] = x_norm
        row[f"kpt{i}_y"] = y_norm
    return row

status_records: List[Dict[str, object]] = []
approved_rows: List[Dict[str, float]] = []

for pred in predictions:
    passed, failed_points = validate_prediction(pred)
    status = {
        "image_name": pred["image_name"],
        "status": "approved" if passed else "rejected",
        "pose_confidence": pred["pose_confidence"],
        "notes": "" if passed else f"Incoherencia punto(s): {', '.join(map(str, failed_points))}",
    }
    status_records.append(status)
    if passed:
        approved_rows.append(prediction_to_row(pred))

for fail in failures:
    status_records.append({
        "image_name": fail["image_name"],
        "status": "inference_failed",
        "pose_confidence": np.nan,
        "notes": fail["reason"],
    })

status_df = pd.DataFrame(status_records)
approved_df = pd.DataFrame(approved_rows)
print(f"Aprobados: {len(approved_df)} / {len(predictions)} predichos")

timestamp = pd.Timestamp.now().strftime("%Y%m%d-%H%M%S")
status_path = OUTPUT_DIR / f"prediction_status_{timestamp}.csv"
status_df.to_csv(status_path, index=False)
print(f"Status guardado en {status_path}")

if not approved_df.empty:
    landmarks_path = OUTPUT_DIR / f"landmarks_approved_{timestamp}.csv"
    approved_df.to_csv(landmarks_path, index=False)
    display(approved_df.head())
else:
    print("No hay landmarks aprobados; revisa las tolerancias o las imágenes.")


## Funciones GPA y PCA (equivalentes a `MorphometricAnalysis`)


In [ ]:
def dataframe_to_shapes(df: pd.DataFrame) -> Tuple[List[str], List[np.ndarray]]:
    image_names = df["image_name"].tolist()
    shapes: List[np.ndarray] = []
    for _, row in df.iterrows():
        coords = []
        for i in range(1, KEYPOINTS_TOTAL + 1):
            x = float(row.get(f"kpt{i}_x", 0.0))
            y = float(row.get(f"kpt{i}_y", 0.0))
            coords.append([x, y])
        shapes.append(np.array(coords, dtype=np.float64))
    return image_names, shapes


def centroid(shape: np.ndarray) -> np.ndarray:
    return shape.mean(axis=0)


def centroid_size(shape: np.ndarray) -> float:
    c = centroid(shape)
    diffs = shape - c
    return float(np.sqrt((diffs ** 2).sum()))


def center_and_scale(shape: np.ndarray, scale: bool = True) -> np.ndarray:
    c = centroid(shape)
    centered = shape - c
    if scale:
        cs = centroid_size(shape)
        if cs > 0:
            centered /= cs
    return centered


def rotation_kabsch(target: np.ndarray, reference: np.ndarray) -> np.ndarray:
    C = target.T @ reference
    U, _, Vt = np.linalg.svd(C)
    d = np.linalg.det(U @ Vt)
    sign = 1.0 if d >= 0 else -1.0
    diag = np.diag([1.0, sign])
    return U @ diag @ Vt


def run_gpa(shapes: List[np.ndarray], max_iter: int = 10, tol: float = 1e-9) -> np.ndarray:
    if not shapes:
        raise ValueError("run_gpa: lista vacía")
    aligned = np.array([center_and_scale(s, True) for s in shapes])
    for _ in range(max_iter):
        mean_shape = aligned.mean(axis=0)
        ref = center_and_scale(mean_shape, True)
        new_aligned = []
        max_delta = 0.0
        for s in aligned:
            R = rotation_kabsch(s, ref)
            rotated = s @ R
            max_delta = max(max_delta, np.linalg.norm(rotated - s))
            new_aligned.append(rotated)
        aligned = np.array(new_aligned)
        if max_delta < tol:
            break
    swapped = aligned.copy()
    swapped[..., [0, 1]] = swapped[..., [1, 0]]  # swap X<->Y al estilo geomorph
    return swapped


def run_pca(aligned_shapes: np.ndarray, k: int = 3) -> Dict[str, object]:
    if aligned_shapes.size == 0:
        raise ValueError("run_pca: sin datos")
    n = aligned_shapes.shape[0]
    flat = aligned_shapes.reshape(n, -1)
    mean_vec = flat.mean(axis=0)
    centered = flat - mean_vec
    cov = centered.T @ centered / max(1, (n - 1))
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    k = min(k, eigenvectors.shape[1])
    eigenvalues = eigenvalues[:k]
    eigenvectors = eigenvectors[:, :k]
    scores = centered @ eigenvectors
    trace = float(np.trace(cov))
    explained = eigenvalues / trace if trace > 0 else eigenvalues
    mean_shape = mean_vec.reshape(aligned_shapes.shape[1], 2)
    return {
        "scores": scores,
        "pcs": eigenvectors,
        "eigenvalues": eigenvalues,
        "explained_ratio": explained,
        "mean_shape": mean_shape,
    }


## Ejecutar GPA + PCA y guardar resultados


In [ ]:
if approved_df.empty:
    print("Sin especímenes aprobados; omitiendo GPA/PCA.")
    gpa_aligned = None
    pca_result = None
else:
    image_names, shapes = dataframe_to_shapes(approved_df)
    gpa_aligned = run_gpa(shapes)
    pca_result = run_pca(gpa_aligned, k=3)
    scores = pd.DataFrame(pca_result["scores"], columns=["PC1", "PC2", "PC3"][: pca_result["scores"].shape[1]])
    scores.insert(0, "image_name", image_names)
    variance = pd.DataFrame({
        "component": [f"PC{i+1}" for i in range(len(pca_result["eigenvalues"]))],
        "eigenvalue": pca_result["eigenvalues"],
        "explained_ratio": pca_result["explained_ratio"],
    })
    scores_path = OUTPUT_DIR / f"pca_scores_{timestamp}.csv"
    var_path = OUTPUT_DIR / f"pca_variance_{timestamp}.csv"
    scores.to_csv(scores_path, index=False)
    variance.to_csv(var_path, index=False)
    print(f"Scores guardados en {scores_path}")
    print(f"Varianzas guardadas en {var_path}")
    display(scores.head())


## Wireframes (±2DE), tabla de scores y morfoespacio


In [ ]:
def vector_to_shape(vec: np.ndarray) -> np.ndarray:
    return vec.reshape(KEYPOINTS_TOTAL, 2)


def deform_shape(mean_flat: np.ndarray, loading: np.ndarray, t: float) -> np.ndarray:
    return vector_to_shape(mean_flat + loading * t)


def to_plot_coords(shape: np.ndarray) -> np.ndarray:
    swapped = shape.copy()
    swapped[:, [0, 1]] = swapped[:, [1, 0]]
    swapped[:, 1] *= -1  # invertir Y para asemejar la vista de Flutter
    return swapped


def plot_wireframe(ax, shape: np.ndarray, color: str, label: str, draw_points: bool = True):
    pts = to_plot_coords(shape)
    xs, ys = pts[:, 0], pts[:, 1]
    ax.plot(xs.tolist() + [xs[0]], ys.tolist() + [ys[0]], color=color, linewidth=1.5, label=label)
    if draw_points:
        ax.scatter(xs, ys, color=color, s=12)
    ax.set_aspect('equal')
    ax.axis('off')


if pca_result is None:
    print("Sin PCA para visualizar")
else:
    mean_shape = pca_result["mean_shape"]
    mean_flat = mean_shape.reshape(-1)
    pcs = pca_result["pcs"]
    eigenvalues = pca_result["eigenvalues"]
    explained = pca_result["explained_ratio"]
    comps = min(3, pcs.shape[1])
    fig, axes = plt.subplots(comps, 3, figsize=(12, 4 * comps))
    if comps == 1:
        axes = np.expand_dims(axes, axis=0)
    for idx in range(comps):
        loading = pcs[:, idx]
        lam = max(eigenvalues[idx], 0.0)
        t = 2.0 * math.sqrt(lam)
        minus = deform_shape(mean_flat, loading, -t)
        plus = deform_shape(mean_flat, loading, t)
        plot_wireframe(axes[idx, 0], minus, "#f97316", "-2DE")
        plot_wireframe(axes[idx, 1], mean_shape, "#0f172a", "Media")
        plot_wireframe(axes[idx, 2], plus, "#22c55e", "+2DE")
        axes[idx, 1].set_title(f"PC{idx+1} ({explained[idx]*100:.1f}% var)")
    plt.tight_layout()

    if 'scores' in locals():
        display(scores)
        fig2, ax = plt.subplots(figsize=(6, 6))
        ax.scatter(scores["PC1"], scores["PC2"], color="#2563eb", alpha=0.8)
        ax.axhline(0, color="gray", linewidth=1)
        ax.axvline(0, color="gray", linewidth=1)
        ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}% var)")
        ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}% var)" if explained.shape[0] > 1 else "PC2")
        if len(scores) <= 25:
            for _, row in scores.iterrows():
                ax.annotate(row["image_name"], (row["PC1"], row["PC2"]), fontsize=8)
        ax.set_title("Morfoespacio PC1 vs PC2")
        plt.show()


## Próximos pasos sugeridos
1. Ajusta `IMAGES_DIR` o las tolerancias para repetir el experimento con otros lotes.
2. Copia los CSV generados (landmarks, scores, varianzas) para compararlos contra tus scripts en R.
3. Modifica el bloque de visualización para agregar PCs adicionales o exportar las figuras como PNG (`plt.savefig`).
